# O2 A Core Primitive Bottleneck Demo

This notebook isolates the LABOS primitive wall behind the O2 A per-spectrum bottleneck. It runs the repo's Zig kernel benchmark, parses the nanoseconds-per-call values, and multiplies them by the measured O2 A matrix call counts.

Code sites:

- `tests/perf/labos_kernel_bench.zig`
- `src/forward_model/radiative_transfer/labos/layers.zig`
- `src/forward_model/radiative_transfer/labos/matrix.zig`

In [1]:
import re
import subprocess
from dataclasses import dataclass
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "build.zig").exists() and (path / "src").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the zdisamar repository")


REPO_ROOT = find_repo_root(Path.cwd())


@dataclass(frozen=True)
class KernelTiming:
    name: str
    ns_per_call: float


@dataclass(frozen=True)
class PrimitiveCall:
    label: str
    kernel: str
    calls: int

    def milliseconds(self, timings: dict[str, KernelTiming]) -> float:
        return self.calls * timings[self.kernel].ns_per_call / 1.0e6

## Measured O2 A Work Counts

These are the core primitive counts for one 701-sample O2 A spectrum after support deduplication. The key fan-out is:

```text
701 nominal wavelengths -> 3874 exact radiance solves
3874 solves -> 120390 Fourier terms
RT-layer doubling -> 8389666 doubling steps
```

In [2]:
SPECTRUM_COUNTS = {
    "nominal_samples": 701,
    "exact_radiance_solves": 3874,
    "fourier_terms": 120390,
    "labos_layers": 5417550,
    "doubled_layers": 1075939,
    "double_steps": 8389666,
}

MATRIX_CALLS = [
    PrimitiveCall("Q = qseries(R * R)", "qseries_12x10", 3_408_299),
    PrimitiveCall("D = T + Q * diag(E) + Q * T", "smulAddSemul3_12", 3_408_299),
    PrimitiveCall("rd = R * D", "smul_12x10", 8_389_666),
    PrimitiveCall("U = R * diag(E) + rd", "semulAdd_12", 8_389_666),
    PrimitiveCall("tu = T * U", "smul_12x10", 8_389_666),
    PrimitiveCall("R_next = R + diag(E) * U + tu", "matAddEsmul3_12", 8_389_666),
    PrimitiveCall("td = T * D", "smul_12x10", 8_389_666),
    PrimitiveCall("T_next = diag(E) * D + T * diag(E) + td", "esmulSemulAdd_12", 8_389_666),
]

print("O2 A support fan-out")
for key, value in SPECTRUM_COUNTS.items():
    print(f"  {key:24s} {value:>10,d}")

fourier_terms_per_solve = (
    SPECTRUM_COUNTS["fourier_terms"] / SPECTRUM_COUNTS["exact_radiance_solves"]
)
print(f"  Fourier terms / solve     {fourier_terms_per_solve:10.3f}")

O2 A support fan-out
  nominal_samples                 701
  exact_radiance_solves         3,874
  fourier_terms               120,390
  labos_layers              5,417,550
  doubled_layers            1,075,939
  double_steps              8,389,666
  Fourier terms / solve         31.076


## Run the Zig Primitive Benchmark

The benchmark uses the actual Zig kernels rather than a reimplementation, so the demo stays focused on the LABOS primitive cost.

In [3]:
BENCH_RE = re.compile(r"^(?P<name>[a-zA-Z0-9_]+): .* ns_per_call=(?P<ns>[0-9.]+)")


def run_zig_bench() -> dict[str, KernelTiming]:
    completed = subprocess.run(
        ["zig", "build", "bench"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    output = completed.stdout + completed.stderr
    timings: dict[str, KernelTiming] = {}
    for line in output.splitlines():
        match = BENCH_RE.match(line.strip())
        if not match:
            continue
        name = match.group("name")
        timings[name] = KernelTiming(
            name=name,
            ns_per_call=float(match.group("ns")),
        )
    return timings


timings = run_zig_bench()
for timing in timings.values():
    print(f"{timing.name:28s} {timing.ns_per_call:10.3f} ns/call")

qseries_12x10                   534.280 ns/call
qseries_nonzero_12x10           520.763 ns/call
smul_12x10                      154.137 ns/call
smulAddSemul3_12                168.300 ns/call
matAddSemul3_12                  88.653 ns/call
matAddEsmul3_12                  99.491 ns/call
semulAdd_12                      67.719 ns/call
esmulSemulAdd_12                 94.018 ns/call


## Reconstruct the Doubling Primitive Cost

The doubling loop in `layers.zig` applies the same small set of matrix primitives millions of times. This cell multiplies isolated kernel timings by the measured O2 A call counts.

In [6]:
rows = []
total_ms = 0.0
for primitive in MATRIX_CALLS:
    elapsed_ms = primitive.milliseconds(timings)
    total_ms += elapsed_ms
    rows.append((primitive.label, primitive.kernel, primitive.calls, elapsed_ms))

print("Primitive contribution estimate\n")
print(f"{'operation':46s} {'kernel':24s} {'calls':>12s} {'worker_ms':>12s}")
for label, kernel, calls, elapsed_ms in rows:
    print(f"{label:46s} {kernel:24s} {calls:12,d} {elapsed_ms:12.3f}")
print(f"{'\ntotal matrix primitive estimate':72s} {total_ms:12.3f}")

Primitive contribution estimate

operation                                      kernel                          calls    worker_ms
Q = qseries(R * R)                             qseries_12x10               3,408,299     1820.986
D = T + Q * diag(E) + Q * T                    smulAddSemul3_12            3,408,299      573.617
rd = R * D                                     smul_12x10                  8,389,666     1293.158
U = R * diag(E) + rd                           semulAdd_12                 8,389,666      568.140
tu = T * U                                     smul_12x10                  8,389,666     1293.158
R_next = R + diag(E) * U + tu                  matAddEsmul3_12             8,389,666      834.696
td = T * D                                     smul_12x10                  8,389,666     1293.158
T_next = diag(E) * D + T * diag(E) + td        esmulSemulAdd_12            8,389,666      788.780
total matrix primitive estimate                                              8465.692

## Interpretation

The reconstruction should land in the same band as the measured doubling bucket. The exact number varies with CPU load and compiler output, but the result demonstrates the wall:

```text
millions of 12x10 / 12x12 f64 kernels
  * under 3874 exact radiance solves
  * under 120390 Fourier terms
  * under 8389666 doubling steps
```

Even when each primitive is sub-microsecond, the product is seconds of worker time. That is the bottleneck this research folder is documenting.

In [5]:
MEASURED_DOUBLING_MS = 8_347.130
ratio = total_ms / MEASURED_DOUBLING_MS
print(f"estimated matrix primitive worker time: {total_ms:.3f} ms")
print(f"measured doubling worker time:          {MEASURED_DOUBLING_MS:.3f} ms")
print(f"estimate / measured:                    {ratio:.3f}")

estimated matrix primitive worker time: 8465.692 ms
measured doubling worker time:          8347.130 ms
estimate / measured:                    1.014
